# Synthetic Data Generator Pipeline
### Step 6 : Producer Queue Manager


In [ ]:
from utils.postgres_util import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.producer_queue_manager import (
    ensure_simulation_state_control_table,
    upsert_simulation_state_control,
    read_simulation_state_control,
    get_send_queue_status_counts,
)

In [10]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

SIMULATION_STATE_CONTROL_TABLE_NAME = str("simulation_state_control")
SEND_QUEUE_TABLE_NAME = str("synthetic_sensor_messages_send_queue")

CONTROL_OWNER_ROLE = str("kafka_producer")
APPLY_OWNER_AND_GRANTS = bool(True)

PRODUCER_TOPIC = str("pump.telemetry.synthetic")
PRODUCER_BATCH_SIZE = int(5200)
PRODUCER_POLL_SECONDS = float(0.0)
MAX_SEND_ATTEMPTS = int(3)

---

In [11]:

engine = get_engine_from_env()


---

In [12]:
control_table_name = ensure_simulation_state_control_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_STATE_CONTROL_TABLE_NAME,
    owner_role=CONTROL_OWNER_ROLE,
    apply_owner_and_grants=APPLY_OWNER_AND_GRANTS,
)

print("Ensured control table:", control_table_name)

Ensured control table: simulation_state_control


---

In [13]:
upsert_simulation_state_control(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    is_enabled=True,
    producer_topic=PRODUCER_TOPIC,
    producer_batch_size=PRODUCER_BATCH_SIZE,
    producer_poll_seconds=PRODUCER_POLL_SECONDS,
    max_send_attempts=MAX_SEND_ATTEMPTS,
    schema=SCHEMA,
    table_name=SIMULATION_STATE_CONTROL_TABLE_NAME,
)

print("Upserted control row.")

Upserted control row.


---

In [14]:
control_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        is_enabled,
        producer_topic,
        producer_batch_size,
        producer_poll_seconds,
        max_send_attempts,
        updated_at,
        created_at
    FROM {SCHEMA}.{SIMULATION_STATE_CONTROL_TABLE_NAME}
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(control_dataframe)

,dataset_id,run_id,is_enabled,producer_topic,producer_batch_size,producer_poll_seconds,max_send_attempts,updated_at,created_at
0,pump_synthetic_v1,premelt_run_001,True,pump.telemetry.synthetic,5200,0.0,3,2026-04-04 21:05:20.085005+00:00,2026-04-04 21:05:20.085005+00:00


---

In [15]:
queue_status_dataframe = get_send_queue_status_counts(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

display(queue_status_dataframe)

,queue_status,row_count
0,pending,3744000


---

In [16]:
queue_preview_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count
    FROM {SCHEMA}.{SEND_QUEUE_TABLE_NAME}
    GROUP BY queue_status
    ORDER BY queue_status
    """
)

display(queue_preview_dataframe)

,queue_status,row_count
0,pending,3744000


---

In [17]:
# Dispose Engine
engine.dispose()